# Imports

In [1]:
from sklearn.datasets import fetch_covtype, load_breast_cancer
from sklearn.tree import DecisionTreeClassifier

from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

from tools.tree_reader.decision_path_extractor import (
    DecisionPathExtractor
)

from tools.tree_reader.feature_importance_extractor import (
    FeatureImportanceExtractor
)

In [2]:
import os

os.environ.pop(
    "SSL_CERT_FILE",
    None
)

'D:\\combined-cabundle.pem'

# Utils

In [3]:
def print_response(response):
    print("=" * 80)
    print("TOOLS USED")
    print("=" * 80)

    for msg in response["messages"]:

        if hasattr(msg, "tool_calls"):

            for tool in msg.tool_calls:
                print(f"✓ {tool['name']}")

    print("\n" + "=" * 80)
    print("AI RESPONSE")
    print("=" * 80)

    print(response["messages"][-1].content)

# Guardrail

In [4]:
ALLOWED_TOPICS = [
    "prediction",
    "predict",
    "classification",
    "classified",
    "decision tree",
    "decision path",
    "feature importance",
    "important features",
    "model",
    "machine learning",
    "explain",
    "explanation",
    "why was",
    "why did",
    "tree"
]


def scope_guardrail(question: str):

    question = question.lower()

    for topic in ALLOWED_TOPICS:

        if topic in question:
            return True

    return False

In [5]:
print(
    scope_guardrail(
        "Why was this record classified?"
    )
)

True


In [6]:
print(
    scope_guardrail(
        "Who is Virat Kohli?"
    )
)

False


# Load Dataset

In [7]:
data = load_breast_cancer()

X = data.data
y = data.target

print(X.shape)

(569, 30)


# Train Model

In [8]:
model = DecisionTreeClassifier(
    max_depth=6,
    random_state=42
)

model.fit(X, y)

print("Model trained")

Model trained


# Tools

In [9]:
@tool
def predict_tool():
    """
    Predict class for current sample.
    """

    prediction = model.predict(
        sample.reshape(1, -1)
    )[0]
    
    if (
        hasattr(data, "target_names")
        and len(data.target_names) > prediction
    ):
        prediction_name = data.target_names[prediction]
    else:
        prediction_name = str(prediction)

    return prediction_name

In [10]:
@tool
def decision_path_tool():
    """
    Return decision path.
    """
    

    extractor = (
        DecisionPathExtractor(
            model,
            data.feature_names if hasattr(data, "feature_names") else None
        )
    )

    return extractor.extract_path(
        sample
    )

In [11]:
@tool
def feature_importance_tool():
    """
    Return top features.
    """

    extractor = (
        FeatureImportanceExtractor(
            model,
            data.feature_names if hasattr(data, "feature_names") else None
        )
    )

    return (
        extractor
        .get_top_features(5)
        .to_dict(
            orient="records"
        )
    )

In [12]:
@tool
def recommendation_tool():
    """
    Suggest feature changes that may alter the prediction.
    """

    extractor = DecisionPathExtractor(
        model,
        data.feature_names if hasattr(data, "feature_names") else None
    )

    path = extractor.extract_path(
        sample
    )

    recommendations = []

    for rule in path["path"]:

        feature = rule["feature"]
        value = rule["value"]
        threshold = rule["threshold"]

        recommendations.append(
            {
                "feature": feature,
                "current_value": value,
                "target_value": threshold
            }
        )

    return recommendations

# Create Agent

In [13]:
llm = ChatOpenAI(
    model="gpt-4o"
)

agent = create_agent(
    llm,
    [
        predict_tool,
        decision_path_tool,
        feature_importance_tool,
        recommendation_tool
    ]
)

In [14]:
feature_names = [
    f"feature_{i}"
    for i in range(X.shape[1])
]

In [15]:
sample = X[0]

# Query 1

In [16]:
question = (
    "What are the top 5 important features?"
)

if scope_guardrail(question):

    response = agent.invoke(
        {
            "messages": [
                (
                    "user",
                    question
                )
            ]
        }
    )

    print_response(response)

else:

    print(
        "❌ Out of scope question."
    )

TOOLS USED
✓ feature_importance_tool

AI RESPONSE
The top 5 important features are:

1. **Worst radius** with an importance of 0.70
2. **Worst concave points** with an importance of 0.11
3. **Worst texture** with an importance of 0.06
4. **Concavity error** with an importance of 0.03
5. **Mean texture** with an importance of 0.03


# Query 2

In [17]:
response = agent.invoke(
    {
        "messages": [
            (
                "user",
                "Why was this record classified?"
            )
        ]
    }
)

print_response(response)

TOOLS USED
✓ predict_tool
✓ decision_path_tool
✓ feature_importance_tool
✓ recommendation_tool

AI RESPONSE
The record was classified as "malignant." Here is a detailed explanation of why:

1. **Decision Path:**
   - The decision path shows the steps in the decision-making process:
     - **Node 0:** The "worst radius" feature has a value of 25.38, which is greater than the threshold of 16.7950, leading down the path that indicates malignancy.
     - **Node 30:** The "mean texture" feature has a value of 10.38, which is less than or equal to the threshold of 16.1100, continuing along the malignancy path.
     - **Node 31:** The "concavity error" feature has a value of 0.05373, which is greater than the threshold of 0.0337, further indicating malignancy.

2. **Top Features Influencing Classification:**
   - **Worst Radius:** This feature has the highest importance score at 0.7006, signifying it plays a critical role in the classification.
   - Other influential features include "worst c

# Query 3

In [18]:
response = agent.invoke(
    {
        "messages": [
            (
                "user",
                "Explain the prediction in business language."
            )
        ]
    }
)

print_response(response)

TOOLS USED
✓ decision_path_tool
✓ feature_importance_tool

AI RESPONSE
The prediction leverages several key factors, primarily focusing on the "worst radius," which is the most influential feature with an importance score of approximately 70%. In this scenario, the decision path reveals that the current sample's worst radius is measured at 25.38, exceeding a crucial threshold of 16.795. This condition initiates the first significant branching in the decision tree.

Following this, the model evaluates the "mean texture," measured at 10.38, which is below a set threshold of 16.11. This further narrows down the decision path.

Finally, the "concavity error" is considered, with a value of 0.05373, surpassing the critical threshold of 0.0337. This condition solidifies the pathway to the leaf node associated with the prediction.

In essence, the model's decision is predominantly driven by the "worst radius" and "concavity error," with "mean texture" acting as a secondary filter, highlighting

# Query 4

In [19]:
question = (
    "Who is Virat Kohli?"
)

if scope_guardrail(question):

    response = agent.invoke(
        {
            "messages": [
                (
                    "user",
                    question
                )
            ]
        }
    )

    print_response(response)

else:

    print(
        "❌ Out of scope question."
    )

❌ Out of scope question.


# Query 4

In [20]:
question = (
    "How can I change this prediction?"
)

if scope_guardrail(question):

    response = agent.invoke(
        {
            "messages": [
                (
                    "user",
                    question
                )
            ]
        }
    )

    print_response(response)

else:

    print(
        "❌ Out of scope question."
    )

TOOLS USED
✓ recommendation_tool
✓ decision_path_tool

AI RESPONSE
To change this prediction, you can consider adjusting the following features:

1. **Worst Radius**: Currently, it is at 25.38. Adjusting it to a target value of 16.795 might help alter the prediction.
2. **Mean Texture**: Currently, it is at 10.38. Changing it to a target value of 16.11 could also contribute to altering the prediction.
3. **Concavity Error**: Currently, it is at 0.05373. Aiming for a target value of 0.0337 could be beneficial.

The decision path indicates that:
- The worst radius being greater than 16.795 is a significant factor.
- The mean texture being less than or equal to 16.1100 is another key factor.
- The concavity error being greater than 0.0337 also impacts the prediction.

Adjusting these features towards the target values suggested may result in a different prediction.
